# Airport Statistics EDA notebook

# 0. Config

In [ ]:
import pandas as pd

# 1. Data

In [9]:
list_dts = [2004 + i for i in range(20)]
list_dts

[2004,
 2005,
 2006,
 2007,
 2008,
 2009,
 2010,
 2011,
 2012,
 2013,
 2014,
 2015,
 2016,
 2017,
 2018,
 2019,
 2020,
 2021,
 2022,
 2023]

In [54]:
path = "../../data/Table 6 Ranking of Major Airport On-Time Departure Performance Year-to-date December 2003-Dec 2023.xlsx"
dic_years = {}
for dt in list_dts:
    dic_years[dt] = pd.read_excel(path, engine="openpyxl", sheet_name=f"{dt}")

# 2. Proccessing

In [55]:
df_airports = pd.DataFrame()

for dt in list_dts:
    df_aux = dic_years[dt].copy().reset_index(drop=True)
    df_aux = df_aux.iloc[2:, 3:]
    df_aux.columns = ["rank","airport", "departure_performance"]
    df_aux["year"] = dt
    df_aux = df_aux[~df_aux["rank"].isna()]
    df_aux = df_aux[df_aux["departure_performance"] != "%"]
    df_aux["departure_performance"] = df_aux["departure_performance"].astype(str).str.replace("%","").astype(float)
    df_aux["rank"] = df_aux["rank"].astype(int)

    df_aux["airport_code"] = df_aux["airport"].str.extract(r"\((.*?)\)")
    print(dt, df_aux.shape)

    df_airports = pd.concat([df_airports, df_aux])

2004 (31, 5)
2005 (33, 5)
2006 (31, 5)
2007 (32, 5)
2008 (32, 5)
2009 (31, 5)
2010 (29, 5)
2011 (29, 5)
2012 (29, 5)
2013 (29, 5)
2014 (29, 5)
2015 (29, 5)
2016 (29, 5)
2017 (30, 5)
2018 (30, 5)
2019 (30, 5)
2020 (30, 5)
2021 (30, 5)
2022 (30, 5)
2023 (30, 5)


In [56]:
df_airports.shape

(603, 5)

# 3. Analysis

In [57]:
df_airports.describe()

,rank,departure_performance,year
count,603.000000,603.000000,603.000000
mean,15.597015,79.563571,2013.388060
std,8.744915,4.664444,5.827132
min,1.000000,60.970000,2004.000000
25%,8.000000,76.660000,2008.000000
50%,16.000000,79.810000,2013.000000
75%,23.000000,82.810000,2018.000000
max,33.000000,92.110000,2023.000000


In [60]:
df_airports.groupby("airport_code").agg({"year":"nunique","departure_performance":"mean", "rank":"mean"}).sort_values("departure_performance", ascending=False)

,year,departure_performance,rank
airport_code,,,
HNL,3,90.060000,1.000000
SLC,20,86.201178,1.900000
PDX,17,85.960917,2.882353
MSP,20,83.574253,5.800000
SEA,20,83.372937,7.800000
PIT,3,82.361731,8.666667
SAN,20,82.351428,8.550000
DTW,20,82.128557,9.150000
DCA,20,82.052561,9.650000


# 4. Export

In [64]:
path = "../../outputs/proc/"

In [62]:
df_airports.dtypes

rank                       int64
airport                   object
departure_performance    float64
year                       int64
airport_code              object
dtype: object

In [69]:
df_dd = df_airports.dtypes.to_frame().reset_index()
df_dd.columns = ["column", "dtype"]
df_dd.to_csv(f"{path}airport_departure_stats_data_dictionary.csv", index=False)

In [65]:
df_airports.to_csv(f"{path}airport_departure_stats_proc.csv", index=False)